In [2]:
import spacy
import json

In [6]:
# Install the spaCy model
!python -m spacy download en_core_web_lg

# Load pre-trained model with existing labels (PERSON, ORG, etc.)
nlp = spacy.load("en_core_web_lg")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Data Preparation - Label your data - Use "Label Studio" Open Source Project

https://labelstud.io/

In [3]:
# Load spaCy training data from converted JSON
with open("/content/claim_training.json", "r") as f:
    file_content = f.read()
    print(file_content)
    TRAIN_DATA = json.loads(file_content)

[
    ["Patient Name: Emily Johnson", {"entities": [[14, 27, "PATIENT_NAME"]]}],
    ["Spouse Name: Alex Johnson", {"entities": [[13, 25, "SPOUSE_NAME"]]}],
    ["Patient ID: A123456789", {"entities": [[12, 22, "PATIENT_ID"]]}],
    ["Insurance Company: HealthSecure Inc.", {"entities": [[20, 39, "INSURANCE_COMPANY"]]}],
    ["Policy Number: HS-789456123", {"entities": [[15, 27, "POLICY_NUMBER"]]}],
    ["Policy Holder: Alex Johnson", {"entities": [[15, 27, "POLICY_HOLDER"]]}],
    ["Sum Insured: $100,000", {"entities": [[13, 21, "SUM_INSURED"]]}],
    ["Hospital Name: St. Mary's Women's Hospital", {"entities": [[15, 45, "HOSPITAL_NAME"]]}],
    ["Admission Date: 2025-04-18", {"entities": [[16, 26, "ADMISSION_DATE"]]}],
    ["Discharge Date: 2025-04-21", {"entities": [[16, 26, "DISCHARGE_DATE"]]}],
    ["Hospital NPI: 1234567890", {"entities": [[14, 24, "HOSPITAL_NPI"]]}],
    ["Hospital TIN: 98-7654321", {"entities": [[14, 24, "HOSPITAL_TIN"]]}],
    ["Total Claimed Amount: $6300", {"e

In [4]:
TRAIN_DATA[4][0].index("HS-789456123")


15

# Converting training data to SpaCy Docbin format

In [7]:
from spacy.tokens import DocBin
from tqdm import tqdm
from spacy.util import filter_spans


doc_bin = DocBin()

for text, annotations in tqdm(TRAIN_DATA):
    doc = nlp.make_doc(text)
    ents = []

    for ent in annotations.get("entities"):
      start,end,label = ent[0],ent[1],ent[2]
      span = doc.char_span(start,end,label=label,alignment_mode="contract")
      if span is None:
        print("Skipping entity")
      else:
        ents.append(span)
    filtered_ents = filter_spans(ents)
    doc.ents = filtered_ents
    doc_bin.add(doc)


doc_bin.to_disk("train.spacy")

100%|██████████| 14/14 [00:00<00:00, 2628.48it/s]

Skipping entity
Skipping entity


Download base_config.cfg file from spacy website

https://spacy.io/usage/training#quickstart

# Initialize spaCy with the config files

In [8]:
!python -m spacy init fill-config base_config.cfg config.cfg

✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


# Train spaCy model

In [9]:
!python -m spacy train config.cfg --output ./ --paths.train ./train.spacy --paths.dev ./train.spacy

ℹ Saving to output directory: .
ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     72.44    0.00    0.00    0.00    0.00
200     200         25.55   1957.11  100.00  100.00  100.00    1.00
400     400          0.00      0.00  100.00  100.00  100.00    1.00
600     600          0.00      0.00  100.00  100.00  100.00    1.00
800     800          0.00      0.00  100.00  100.00  100.00    1.00
1000    1000          0.00      0.00  100.00  100.00  100.00    1.00
1200    1200          0.00      0.00  100.00  100.00  100.00    1.00
1400    1400          0.00      0.00  100.00  100.00 

# Load the fine-tuned model for downstream work

In [10]:
import spacy

# Load your fine-tuned model
nlp_ner = spacy.load("model-best")

# Test with text that includes both default and custom entities
test_text = """
Patient Name: Emily Johnson
Spouse Name: Alex Johnson
Patient ID: A123456789
Date of Birth: 1992-08-15
Gender: Female
Contact Number: +1-212-555-1234
Email: alex.johnson@example.com
Insurance Company: HealthSecure Inc.
Policy Number: HS-789456123
Policy Holder: Alex Johnson
Relationship to Patient: Spouse
Coverage Type: Family Floater
Sum Insured: $100,000
Hospital Name: St. Mary's Women's Hospital
Admission Date: 2025-04-18
Discharge Date: 2025-04-21
Treatment Type: Cesarean Delivery
Total Claimed Amount: $6300

 """

doc = nlp_ner(test_text)
#print(doc.ents)

# Print all detected entities
for ent in doc.ents:
    print(f"This is label: {ent.label_} and this is value : {ent.text}")
    #print(ent.label_, "-", ent.text)

This is label: PATIENT_NAME and this is value : Emily Johnson
This is label: SPOUSE_NAME and this is value : Alex Johnson
This is label: PATIENT_ID and this is value : A123456789
This is label: SIGNATURE_FIELD and this is value : Date of Birth: 1992-08-15
Gender: Female
Contact Number: +1-212-555-1234
Email: alex.johnson@example.com
Insurance Company: HealthSecure Inc.
Policy Number: HS-789456123
Policy Holder: Alex Johnson
Relationship to Patient: Spouse
Coverage Type: Family Floater
Sum Insured: $100,000

This is label: ADMISSION_DATE and this is value : 2025-04-18
This is label: DISCHARGE_DATE and this is value : 2025-04-21
This is label: CLAIMED_AMOUNT and this is value : $6300


In [12]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 97.1 MB/s eta 0:00:00


In [15]:
import fitz  # PyMuPDF library
file_text = ""

try:
    # Open the PDF file in binary mode
    with fitz.open("/content/Claim_Form_Emily_Johnson.pdf") as doc:
        # Extract text from each page
        for page in doc:
            file_text += page.get_text()

    # Now 'text' contains the extracted text from the PDF, which you can prin

except FileNotFoundError:
    print("Error: The file was not found at the specified path.")
except Exception as e:
    print(f"An error occurred: {e}")

print(file_text)

Health Insurance Claim Form
Claimant Information:
----------------------
Patient Name: Emily Johnson
Spouse Name: Alex Johnson
Patient ID: A123456789
Date of Birth: 1992-08-15
Gender: Female
Contact Number: +1-212-555-1234
Email: alex.johnson@example.com
Policy Information:
--------------------
Insurance Company: HealthSecure Inc.
Policy Number: HS-789456123
Policy Holder: Alex Johnson
Relationship to Patient: Spouse
Coverage Type: Family Floater
Sum Insured: $100,000
Hospitalization Details:
-------------------------
Hospital Name: St. Mary's Women's Hospital
Admission Date: 2025-04-18
Discharge Date: 2025-04-21
Page 1
Health Insurance Claim Form
Treatment Type: Cesarean Delivery
Total Claimed Amount: $6300
Bank Details (for reimbursement):
----------------------------------
Bank Name: Chase Bank
Account Holder Name: Alex Johnson
Account Number: XXXX-XXXX-5678
IFSC/SWIFT Code: CHASUS33XXX
Declaration:
-------------
I hereby declare that the information provided above is true and corre

In [16]:
doc = nlp_ner(file_text)
#print(doc.ents)

# Print all detected entities
for ent in doc.ents:
    print(f"This is label: {ent.label_} and this is value : {ent.text}")
    #print(ent.label_, "-", ent.text)

This is label: PATIENT_NAME and this is value : Emily Johnson
This is label: SPOUSE_NAME and this is value : Alex Johnson
This is label: PATIENT_ID and this is value : A123456789
This is label: SIGNATURE_FIELD and this is value : Date of Birth: 1992-08-15
Gender: Female
Contact Number: +1-212-555-1234
Email: alex.johnson@example.com
Policy Information:
--------------------
Insurance Company: HealthSecure Inc.
Policy Number: HS-789456123
Policy Holder: Alex Johnson
Relationship to Patient: Spouse
Coverage Type: Family Floater
Sum Insured: $100,000

This is label: ADMISSION_DATE and this is value : 2025-04-18
This is label: DISCHARGE_DATE and this is value : 2025-04-21
This is label: CLAIMED_AMOUNT and this is value : $6300
This is label: SPOUSE_NAME and this is value : Alex Johnson
This is label: HOSPITAL_TIN and this is value : XXXX-XXXX-5678
This is label: PATIENT_NAME and this is value : CHASUS33XXX

This is label: SIGNATURE_FIELD and this is value : Signature of Policy Holder: ___
T